**CI twin of `ch05-logistic-regression.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
import matplotlib.pyplot as plt

df = load_csv("penguins")
two = df[df["species"].isin(["Adelie", "Gentoo"])].dropna(
    subset=["flipper_length_mm", "body_mass_g",
            "bill_length_mm", "bill_depth_mm"])
print(two["species"].value_counts())

fig, ax = plt.subplots(figsize=(5.5, 2.6))
for species, level in [("Adelie", 0), ("Gentoo", 1)]:
    sub = two[two["species"] == species]
    ax.scatter(sub["flipper_length_mm"], [level] * len(sub),
               s=10, alpha=0.4, label=species)
ax.set_xlabel("flipper length (mm)")
ax.set_yticks([0, 1], ["Adelie", "Gentoo"])
ax.legend(loc="center right")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd

y = (two["species"] == "Gentoo").astype(int)

line = LinearRegression().fit(two[["flipper_length_mm"]], y)
tries = pd.DataFrame({"flipper_length_mm": [170.0, 200.0, 235.0]})
for mm, val in zip(tries["flipper_length_mm"], line.predict(tries)):
    print(f"flipper {mm:.0f} mm -> species {val:.2f}")

In [ ]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

zs = [z / 10 for z in range(-80, 81)]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(zs, [sigmoid(z) for z in zs])
ax.axhline(0.5, ls=":", lw=1)
ax.axvline(0, ls=":", lw=1)
ax.set_xlabel("z  (the line's score)")
ax.set_ylabel("sigmoid(z)")
plt.show()
print(f"sigmoid(-2)={sigmoid(-2):.4f}  sigmoid(0)={sigmoid(0):.1f}  "
      f"sigmoid(2)={sigmoid(2):.4f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

Xtr, Xte, ytr, yte = train_test_split(
    two[["flipper_length_mm"]], y,
    test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression()
clf.fit(Xtr, ytr)

probe = pd.DataFrame({"flipper_length_mm": [181.0, 210.0, 230.0]})
for mm, p in zip(probe["flipper_length_mm"], clf.predict_proba(probe)[:, 1]):
    print(f"flipper {mm:.0f} mm -> P(Gentoo) = {p:.3f}")
print(f"\nlearned w = {clf.coef_[0][0]:.3f}, b = {clf.intercept_[0]:.3f}")
print(f"P = 0.5 exactly at flipper = {-clf.intercept_[0] / clf.coef_[0][0]:.1f} mm")

In [ ]:
pred = clf.predict(Xte)
print(f"held-out accuracy: {accuracy_score(yte, pred):.3f}")

In [ ]:
hard = df[df["species"].isin(["Adelie", "Chinstrap"])].dropna(
    subset=["bill_length_mm"])
yh = (hard["species"] == "Chinstrap").astype(int)
hard_Xtr, hard_Xte, hard_ytr, hard_yte = train_test_split(
    hard[["bill_length_mm"]], yh,
    test_size=0.25, random_state=42, stratify=yh)

chin = LogisticRegression().fit(hard_Xtr, hard_ytr)
probs = chin.predict_proba(hard_Xte)[:, 1]
print(f"held-out penguins: {len(hard_yte)}, "
      f"truly Chinstrap: {int(hard_yte.sum())}\n")

for t in (0.3, 0.5, 0.8):
    called = probs >= t
    missed = int((~called & (hard_yte == 1)).sum())
    false_alarm = int((called & (hard_yte == 0)).sum())
    print(f"threshold {t}: called Chinstrap {int(called.sum()):>2}, "
          f"missed real Chinstraps {missed}, false alarms {false_alarm}")

In [ ]:
model = LogisticRegression()
model.fit(Xtr, ytr)

run_tests([
    ("P(Gentoo) for a 210 mm flipper", round(float(
        model.predict_proba(pd.DataFrame({"flipper_length_mm": [210.0]}))[0, 1]
    ), 3), 0.948),
    ("held-out accuracy", round(
        accuracy_score(yte, model.predict(Xte)), 3), 0.986),
])

In [ ]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def classify(probs, threshold):
    return [1 if p >= threshold else 0 for p in probs]

run_tests([
    ("perfect doubt", sigmoid(0), 0.5),
    ("confident yes", round(sigmoid(2), 4), 0.8808),
    ("confident no", round(sigmoid(-2), 4), 0.1192),
    ("mirror symmetry", round(sigmoid(3) + sigmoid(-3), 6), 1.0),
    ("default policy", classify([0.2, 0.5, 0.9], 0.5), [0, 1, 1]),
    ("strict policy", classify([0.2, 0.5, 0.9], 0.8), [0, 0, 1]),
], tol=1e-9)